# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [ ]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [ ]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [ ]:
# 🤖 AGENT FUNCTION (IMPLEMENTED - CUSTOM)

def agent(query: str):
    import re
    import urllib.request
    import urllib.parse
    import json
    
    q = query.strip().lower()
    try:
        # 1. Routing for Calculator Tool
        if "calculate" in q or any(op in q for op in ["+", "-", "*", "/"]):
            expr = query
            if "calculate" in q:
                expr = query[q.find("calculate") + len("calculate"):].strip()
            expr = expr.rstrip("?.! ")
            res = calculator(expr)
            if res == "Error in calculation":
                return {
                    "type": "error",
                    "result": f"Could not compute expression: '{expr}'"
                }
            return {
                "type": "calculation",
                "result": res
            }
            
        # 2. Routing for Keyword Extractor Tool
        elif "keyword" in q:
            target = query
            for prefix_word in ["extract keywords from", "extract keywords", "keywords from", "keywords"]:
                if q.startswith(prefix_word):
                    target = query[len(prefix_word):].strip()
                    break
            target = target.lstrip(": ")
            res = extract_keywords(target)
            return {
                "type": "keywords",
                "result": res
            }
            
        # 3. Routing for General Queries
        else:
            # Intercept greetings first
            greetings = ["hello", "hi", "hii", "hey", "how are you", "how r u"]
            if any(re.search(r'\b' + re.escape(g) + r'\b', q) for g in greetings) or q in ["hii", "hi"]:
                return {
                    "type": "general",
                    "result": "I am operating normally and ready to assist you!"
                }
                
            # Intercept real-time requests
            if "news" in q:
                return {
                    "type": "general",
                    "result": "I don't have real-time access to current news. Please refer to a news website or search engine for the latest updates."
                }
            if "weather" in q:
                return {
                    "type": "general",
                    "result": "I don't have access to live weather data. Please check a weather service for your local forecast."
                }

            # Fetch explanation from Wikipedia API dynamically
            summary = None
            try:
                clean_q = q
                prefixes = ["what is a", "what is an", "what is", "who is", "explain about", "explain me", "explain", "define", "tell me about"]
                for p in prefixes:
                    if clean_q.startswith(p):
                        clean_q = clean_q[len(p):].strip()
                        break
                clean_q = clean_q.rstrip("?.! ")
                
                title = clean_q.title().replace(" ", "_")
                url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{urllib.parse.quote(title)}"
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=4) as response:
                    data = json.loads(response.read().decode("utf-8"))
                    summary = data.get("extract")
            except Exception:
                try:
                    url_raw = f"https://en.wikipedia.org/api/rest_v1/page/summary/{urllib.parse.quote(clean_q.replace(' ', '_'))}"
                    req = urllib.request.Request(url_raw, headers={"User-Agent": "Mozilla/5.0"})
                    with urllib.request.urlopen(req, timeout=4) as response:
                        data = json.loads(response.read().decode("utf-8"))
                        summary = data.get("extract")
                except Exception:
                    pass
                    
            if summary:
                return {
                    "type": "general",
                    "result": summary
                }
                
            # Local Fallback Glossary
            glossary = {
                "deep learning": "A branch of machine learning utilizing multi-layered artificial neural networks.",
                "dl": "Abbreviation for Deep Learning.",
                "machine learning": "A field of AI focused on algorithms that learn from data and improve over time.",
                "ml": "Abbreviation for Machine Learning."
            }
            
            for key, desc in glossary.items():
                if len(key) <= 3:
                    if re.search(r'\b' + re.escape(key) + r'\b', q):
                        return {
                            "type": "general",
                            "result": desc
                        }
                else:
                    if key in q:
                        return {
                            "type": "general",
                            "result": desc
                        }
                                    
            return {
                "type": "general",
                "result": f"Direct response to your query: {query}"
            }
                        
    except Exception as e:
        return {
            "type": "error",
            "result": str(e)
        }


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [ ]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

In [ ]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))